In [ ]:
import torch
from transformers import pipeline
from PIL import Image, ImageDraw, ImageFont

def erkenne_mit_details(bild_pfad):
    # Hardware-Check für M2
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"--- Starte Erkennung auf {device.upper()} ---")

    # Modell laden
    detector = pipeline("object-detection", model="facebook/detr-resnet-50", device=device)

    try:
        bild = Image.open(bild_pfad).convert("RGB")
    except Exception as e:
        print(f"Fehler beim Öffnen: {e}")
        return

    ergebnisse = detector(bild)
    draw = ImageDraw.Draw(bild)
    
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 18)
    except:
        font = ImageFont.load_default()

    print("\nGefundene Objekte:")
    print("-" * 30)

    for obj in ergebnisse:
        label = obj["label"]
        # Score in Prozent umrechnen
        wahrscheinlichkeit = obj["score"] * 100
        box = obj["box"]
        
        # Konsolen-Ausgabe
        print(f"📍 {label.ljust(15)} | Sicherheit: {wahrscheinlichkeit:>6.2f}%")

        # Nur Objekte mit mehr als 70% Sicherheit im Bild anzeigen (Filter)
        if wahrscheinlichkeit > 70:
            xmin, ymin, xmax, ymax = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
            
            # Rahmen zeichnen
            draw.rectangle([xmin, ymin, xmax, ymax], outline="#00FF00", width=4)
            
            # Text-Hintergrund für bessere Lesbarkeit
            text = f"{label}: {wahrscheinlichkeit:.1f}%"
            text_bbox = draw.textbbox((xmin, ymin), text, font=font)
            draw.rectangle(text_bbox, fill="#00FF00")
            draw.text((xmin, ymin), text, fill="black", font=font)

    # Bild speichern & öffnen
    bild.save("analyse_ergebnis.jpg")
    bild.show()

if __name__ == "__main__":
    erkenne_mit_details("/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/train/Vollstrom1_frame00041_Probe00020_00021.jpg")
    erkenne_mit_details("/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/test/Vollstrom1_frame00016_Probe00008.jpg")

--- Starte Erkennung auf MPS ---


Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Gefundene Objekte:
------------------------------
📍 motorcycle      | Sicherheit:  55.59%
📍 motorcycle      | Sicherheit:  67.14%
--- Starte Erkennung auf MPS ---


Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Gefundene Objekte:
------------------------------


In [2]:
from pathlib import Path
import json, os, yaml


# function to convert COCO annotations to YOLO format
def coco_to_yolo(coco_json, labels_dir):
    labels_dir = Path(labels_dir); labels_dir.mkdir(parents=True, exist_ok=True)
    data = json.load(open(coco_json))
    cats = {c["id"]: i for i, c in enumerate(sorted(data["categories"], key=lambda x: x["id"]))}
    imgs = {img["id"]: img for img in data["images"]}
    anns_by_img = {}
    for ann in data["annotations"]:
        if "bbox" not in ann or ann.get("iscrowd", 0) == 1:
            continue
        anns_by_img.setdefault(ann["image_id"], []).append(ann)

    for img_id, img in imgs.items():
        w, h = img["width"], img["height"]
        out = []
        for ann in anns_by_img.get(img_id, []):
            x, y, bw, bh = ann["bbox"]
            xc, yc = (x + bw / 2) / w, (y + bh / 2) / h
            out.append(f'{cats[ann["category_id"]]} {xc:.6f} {yc:.6f} {bw/w:.6f} {bh/h:.6f}')
        (labels_dir / (Path(img["file_name"]).stem + ".txt")).write_text("\n".join(out))


# use current script directory as base dir
base_dir = "/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data"

# Create YOLO labels for train, val, and test sets
coco_to_yolo(os.path.join(base_dir, "train.json"), os.path.join(base_dir, "train"))
coco_to_yolo(os.path.join(base_dir, "val.json"), os.path.join(base_dir, "val"))
coco_to_yolo(os.path.join(base_dir, "test.json"), os.path.join(base_dir, "test")) 

print("generated YOLO text files in the folders train, val, and test.")


# Load class anames from train.json
with open(os.path.join(base_dir, "train.json"), 'r') as f:
    train_data = json.load(f)

class_names = {i: cat["name"] for i, cat in enumerate(sorted(train_data["categories"], key=lambda x: x["id"]))}


# Create data.yaml for YOLO training (yaml might be not needed depending on the framework)
data_yaml = {
    "path": base_dir,
    "train": "train",
    "val": "val",
    "names": class_names
}

with open(os.path.join(base_dir, "yolo.yaml"), "w") as f:
    yaml.dump(data_yaml, f, allow_unicode=True)

print("generated yaml file at:", os.path.join(base_dir, "yolo.yaml"))


generated YOLO text files in the folders train, val, and test.
generated yaml file at: /Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/yolo.yaml


In [2]:
pip install YOLO ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 5.1 MB/s  0:00:00m 4.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 18.1 MB/s  0:00:028.2 MB/s eta 0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 824.0/824.0 kB 19.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 23.5 MB/s  0:00:01 24.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ultralytics] 4/5 [ultralytics]n]
Note: you may need to restart the kernel to use updated packages.


In [4]:
from ultralytics import YOLO

# 1. Ein vortrainiertes Modell laden
model = YOLO("yolov8n.pt") 

# 2. Training starten (nutzt automatisch die M2 GPU/MPS)
model.train(data="/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/yolo.yaml", epochs=100, imgsz=640)

Ultralytics 8.4.37 🚀 Python-3.13.9 torch-2.11.0 CPU (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/yolo.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False

Matplotlib is building the font cache; this may take a moment.


Overriding model.yaml nc=80 with nc=23

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytic

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x168b20d70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,

In [10]:
from ultralytics import YOLO

# Lade dein speziell trainiertes Modell
model = YOLO("/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/runs/detect/train2/weights/best.pt")

# Nutze es für eine neue Erkennung
ergebnisse = model.predict("/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/test/Vollstrom1_frame00016_Probe00008.jpg")


image 1/1 /Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/test/Vollstrom1_frame00016_Probe00008.jpg: 640x640 12 plastic-foil/bags, 3 beverage-cartons, 2 plastic-cup/pots, 1 metal-lid, 2 paper-others, 90.2ms
Speed: 10.3ms preprocess, 90.2ms inference, 11.0ms postprocess per image at shape (1, 3, 640, 640)


In [19]:
from ultralytics import YOLO

# 1. Pfad zu deinem BESTEN Durchgang
mein_modell_pfad = "/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/runs/detect/train2/weights/best.pt"

# 2. Modell laden
model = YOLO(mein_modell_pfad)

# 3. Vorhersage machen
# source: Dein Bild
# save: Speichert das Bild mit Rahmen automatisch ab
# conf: Zeigt nur Objekte an, die sicherer als 0.5 (50%) sind
results = model.predict(source="/Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/test/Vollstrom1_frame00016_Probe00008.jpg", save=True, conf=0.1)

# Ergebnisse in der Konsole anzeigen
#for result in results:
#    print(result.probs)  # Falls Klassifizierung
#s    print(result.boxes)  # Bei Objekterkennung (Koordinaten & Scores)

for result in results:
    boxes = result.boxes
    for box in boxes:
        # Hier bekommst du alles, was du für deine Anzeige brauchst:
        klasse = model.names[int(box.cls)] # Name des Objekts
        konfidenz = float(box.conf)        # Wahrscheinlichkeit (0.0 bis 1.0)
        koordinaten = box.xyxy             # [xmin, ymin, xmax, ymax]
        
        print(f"Gefunden: {klasse} mit {konfidenz:.2%}")



for result in results:
    # Alle Boxen im aktuellen Bild durchlaufen
    for box in result.boxes:
        # Klasse (ID zu Name)
        class_id = int(box.cls[0])
        label = model.names[class_id]
        
        # Wahrscheinlichkeit
        score = float(box.conf[0])
        
        # Koordinaten extrahieren (Format: [xmin, ymin, xmax, ymax])
        # .tolist() macht aus dem Tensor eine normale Python-Liste
        coords = box.xyxy[0].tolist()
        xmin, ymin, xmax, ymax = coords

        print(f"📦 Objekt: {label}")
        print(f"   Sicherheit: {score:.2%}")
        print(f"   Position:   Oben-Links ({int(xmin)}, {int(ymin)})")
        print(f"               Unten-Rechts ({int(xmax)}, {int(ymax)})")
        print("-" * 30)

# Optional: Bild mit Rahmen anzeigen
result.show()


image 1/1 /Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/data/test/Vollstrom1_frame00016_Probe00008.jpg: 640x640 22 plastic-foil/bags, 5 beverage-cartons, 1 plastic-lid, 2 plastic-cup/pots, 1 metal-lid, 2 paper-others, 88.7ms
Speed: 3.7ms preprocess, 88.7ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /Users/patrickschnepf/Desktop/Master WINF/2 Semester/Projekt DS/PackWISE_dataset_v2/runs/detect/predict10
Gefunden: plastic-foil/bag mit 94.12%
Gefunden: plastic-foil/bag mit 92.77%
Gefunden: plastic-foil/bag mit 90.01%
Gefunden: plastic-foil/bag mit 85.75%
Gefunden: plastic-foil/bag mit 79.35%
Gefunden: plastic-foil/bag mit 78.59%
Gefunden: plastic-cup/pot mit 70.94%
Gefunden: plastic-foil/bag mit 68.26%
Gefunden: plastic-foil/bag mit 64.44%
Gefunden: beverage-carton mit 55.23%
Gefunden: plastic-cup/pot mit 53.85%
Gefunden: paper-other mit 49.96%
Gefunden: metal-lid mit 49.58%
Gefunden: plastic-foil/bag mit 47.85%